# SmolDocling Phase 3 Model
**F26-04 | Track C**

Section 1 deals with:
Runs SmolDocling-256M on three evaluation datasets:
1. **Im2LaTeX-230K** — equation recognition
2. **DocLayNet v1.2** — full-page OCR
3. **PubTables-1M** — table structure recognition

Outputs predictions as CSVs/JSONL for Phase 4 metric computation.



---


Section 2 deals with:




Runs Nougat on
1. **Im2LaTeX-230K** — equation recognition
2. **DocLayNet v1.2** — full-page OCR

Outputs predictions as CSVs/JSONL for Phase 4 metric computation.


---



Section 3 deals with:
Runs TATR on **PubTables-1M** — table structure recognition

Outputs predictions as CSVs/JSONL for Phase 4 metric computation.


##1.1:Install packages and load the model
We need `transformers` for the VLM, `datasets` for HuggingFace data, and `docling_core` because SmolDocling outputs DocTags that we parse with it.

!pip install -q --upgrade --force-reinstall transformers

In [ ]:
!pip install -q "numpy<2"

In [45]:
import os, re, json, torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from pathlib import Path
from datasets import load_dataset

try:
    from transformers import AutoProcessor, AutoModelForImageTextToText as VLMClass
except ImportError:
    from transformers import AutoProcessor, AutoModelForVision2Seq as VLMClass

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'ds4sd/SmolDocling-256M-preview'

print(f'Loading {MODEL_ID}...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = VLMClass.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16).to(DEVICE)
model.eval()
print('Model loaded successfully')

Loading ds4sd/SmolDocling-256M-preview...


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

Model loaded successfully


## 1.2: The inference function

This is the core function that takes an image + a task prompt and returns SmolDocling's DocTags output.

**Why `model.generate()` instead of `model()`:** SmolDocling is autoregressive — it produces output one token at a time. A single forward pass (`model(**inputs)`) only gives you logits for the input, not a generated sequence. `generate()` does the full token-by-token generation loop internally.

.SmolDocling was trained on conversations formatted a specific way (with `<|im_start|>User:` etc.). The processor's `apply_chat_template` adds these special tokens automatically — you can't just feed it a raw prompt string.

In [46]:
def run_smoldocling(image, prompt_text, max_tokens=8192):
    """
    Run SmolDocling on one image with a given task prompt.
    """
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': prompt_text},
        ],
    }]

    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors='pt').to(DEVICE)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_tokens,   # Now caller can set a shorter limit per task
            do_sample=False,
        )

    new_tokens = generated_ids[:, inputs['input_ids'].shape[1]:]
    output_text = processor.batch_decode(new_tokens, skip_special_tokens=False)[0]
    output_text = output_text.replace('<end_of_utterance>', '').strip()
    return output_text



def strip_doctags(text):
    """
    Remove all <tag> wrappers (including <loc_N> position tags) to get plain text.
    Used before computing Edit Distance / BLEU / F1 in Phase 4.
    """
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# Sanity check with a real document image
import requests
from io import BytesIO

url = 'https://huggingface.co/ds4sd/SmolDocling-256M-preview/resolve/main/assets/doc_sample.png'
try:
    img = Image.open(BytesIO(requests.get(url, timeout=10).content)).convert('RGB')
except:
    img = Image.new('RGB', (800, 600), 'white')

test_output = run_smoldocling(img, 'Convert this page to docling.')
print(f'Sanity check (first 300 chars):\n{test_output[:300]}')

Sanity check (first 300 chars):
<doctag><section_header_level_1><loc_100><loc_44><loc_399><loc_54>1.1.1.1.1.1</section_header_level_1>
<text><loc_100><loc_63><loc_399><loc_73>1.1.1.1.1.1</text>
<text><loc_100><loc_82><loc_399><loc_92>1.1.1.1.1.1</text>
<text><loc_100><loc_101><loc_399><loc_111>1.1.1.1.1.1</text>
<text><loc_100><lo


In [47]:
print(f'Image size: {img.size}')
print(f'Image mode: {img.mode}')

Image size: (800, 600)
Image mode: RGB


## 1.3 Dataset 1- Im2LaTeX-230K (equations)

Task: image of a math formula → LaTeX string.

Paper prompt: `Convert formula to LaTeX.` (Table 6 in the appendix).

Targets from the paper (Table 2): Edit Distance 0.11, F1 0.95, BLEU 0.83.

In [7]:

from google.colab import userdata
import os

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()

print('Downloading Im2LaTeX-230K...')
!kaggle datasets download -d gregoryeritsyan/im2latex-230k -p /content/im2latex_data/ --unzip -q

# Paths set up by your Phase 2 preprocessing
LATEX_IMG_DIR = '/content/im2latex_data/PRINTED_TEX_230k/generated_png_images'
FORMULA_FILE = '/content/im2latex_data/PRINTED_TEX_230k/final_png_formulas.txt'
MAP_FILE = '/content/im2latex_data/PRINTED_TEX_230k/corresponding_png_images.txt'

# Load the two parallel text files (same order: line N in one = line N in the other)
with open(FORMULA_FILE) as f:
    formulas = [line.strip() for line in f]
with open(MAP_FILE) as f:
    image_names = [line.strip() for line in f]

print(f'Loaded {len(formulas)} formula/image pairs')

Dataset URL: https://www.kaggle.com/datasets/gregoryeritsyan/im2latex-230k
License(s): CC0-1.0
Loaded 238329 formula/image pairs


In [10]:
# Prevent Colab timeout during long runs
import IPython
IPython.display.Javascript('''
function keepAlive() { document.querySelector("colab-toolbar-button#connect").click(); }
setInterval(keepAlive, 60000);
''')

<IPython.core.display.Javascript object>

In [16]:
#upper cap for test for representative data
N_LATEX_SAMPLES = 50

latex_results = []
for img_name, formula in tqdm(list(zip(image_names, formulas))[:N_LATEX_SAMPLES], desc='Im2LaTeX'):
    img_path = os.path.join(LATEX_IMG_DIR, img_name)
    if not os.path.exists(img_path):
        continue

    # Load image and convert to RGB (matches Phase 2 preprocessing)
    image = Image.open(img_path).convert('RGB')

    # Run the model. The paper uses this exact prompt for equations (Appendix Table 6).
    prediction = run_smoldocling(image, 'Convert formula to LaTeX.', max_tokens=512)

    # Save both the raw prediction and a stripped version for metric computation
    latex_results.append({
        'image': img_name,
        'ground_truth_latex': formula,               # Raw ground truth from dataset
        'prediction_raw': prediction,                # Raw model output (with DocTags)
        'prediction_stripped': strip_doctags(prediction),  # Plain LaTeX for metrics
    })

# Save for Phase 4
df_latex = pd.DataFrame(latex_results)
df_latex.to_csv('/content/smoldocling_latex_results.csv', index=False)
print(f'Saved {len(df_latex)} predictions to smoldocling_latex_results.csv')
df_latex.head(3)

Im2LaTeX:   0%|          | 0/50 [00:00<?, ?it/s]

Saved 50 predictions to smoldocling_latex_results.csv


,image,ground_truth_latex,prediction_raw,prediction_stripped
0,80f1db54ec657ab.png,R _ { 1 2 } K _ { 1 } R _ { 2 1 } d K _ { 2 } ...,<formula><loc_0><loc_0><loc_500><loc_500>R _ {...,R _ { 1 2 } K _ { 1 } R _ { 2 1 } d K _ { 2 } ...
1,4c0c01a5fb03248.png,E _ { n } - E _ { m } = \frac { \lambda ^ { \p...,<formula><loc_0><loc_0><loc_500><loc_500>E _ {...,E _ { n } - E _ { m } = \frac { \lambda ^ { \p...
2,3f55826fd850d77.png,\sigma ^ { 1 } + i \sigma ^ { 2 } = f ( \sigma...,<formula><loc_0><loc_0><loc_500><loc_500>\sigm...,\sigma ^ { 1 } + i \sigma ^ { 2 } = f ( \sigma...


In [17]:
#predictions vs ground truth
for i in range(3):
    print(f'--- Sample {i} ---')
    print(f'GT:   {df_latex.iloc[i].ground_truth_latex[:100]}')
    print(f'Pred: {df_latex.iloc[i].prediction_stripped[:100]}')
    print()

--- Sample 0 ---
GT:   R _ { 1 2 } K _ { 1 } R _ { 2 1 } d K _ { 2 } = d K _ { 2 } R _ { 1 2 } K _ { 1 } R _ { 1 2 } ^ { - 
Pred: R _ { 1 2 } K _ { 1 } R _ { 2 1 } d K _ { 2 } = d K _ { 2 } R _ { 1 2 } K _ { 1 } R _ { 1 2 } ^ { - 

--- Sample 1 ---
GT:   E _ { n } - E _ { m } = \frac { \lambda ^ { \prime } ( n ^ { 2 } y ^ { 2 } - m ^ { 2 } ) } { y ^ { 2
Pred: E _ { n } - E _ { m } = \frac { \lambda ^ { \prime } ( n ^ { 2 } y ^ { 2 } - m ^ { 2 } ) } { y ^ { 2

--- Sample 2 ---
GT:   \sigma ^ { 1 } + i \sigma ^ { 2 } = f ( \sigma ^ { 1 } + i \sigma ^ { 2 } )
Pred: \sigma ^ { 1 } + i \sigma ^ { 2 } = f ( \sigma ^ { 1 } + i \sigma ^ { 2 } )



In [18]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/F26-04/phase3_results
!cp /content/smoldocling_latex_results.csv /content/drive/MyDrive/F26-04/phase3_results/
print('Im2LaTeX results saved to Drive')

Mounted at /content/drive
Im2LaTeX results saved to Drive


## 1.4-  Dataset 2: DocLayNet v1.2 (full-page OCR)

Task: full document page image → complete DocTags representation.

Paper prompt: `Convert this page to docling.` (the headline task from Table 6).

Targets from the paper (Table 2): Edit Distance 0.48, F1 0.80, BLEU 0.58.

In [48]:
N_DOCLAYNET_SAMPLES = 50   # 50 for representative evaluation
IMAGE_DIR = '/content/doclaynet_images'
os.makedirs(IMAGE_DIR, exist_ok=True)

#stream only the test data not training data for smoldocling
ds = load_dataset('docling-project/DocLayNet-v1.2', split='test', streaming=True)

doclaynet_results = []
for i, example in enumerate(tqdm(ds, total=N_DOCLAYNET_SAMPLES, desc='DocLayNet')):
    if i >= N_DOCLAYNET_SAMPLES:
        break

    image = example['image'].convert('RGB')

    # Save the page image so we can reference it later
    img_path = os.path.join(IMAGE_DIR, f'page_{i:05d}.png')
    image.save(img_path)

    # Run model with the full-page conversion prompt
    prediction = run_smoldocling(image, 'Convert this page to docling.', max_tokens=2048)
    # DocLayNet v1.2 examples also carry metadata, saving what's useful
    doclaynet_results.append({
        'index': i,
        'image_path': img_path,
        'prediction_raw': prediction,
        'prediction_stripped': strip_doctags(prediction),
    })

# Save as JSONL so each prediction is on its own line (easy to stream-process in Phase 4)
with open('/content/smoldocling_doclaynet_results.jsonl', 'w') as f:
    for res in doclaynet_results:
        f.write(json.dumps(res) + '\n')

print(f'Saved {len(doclaynet_results)} page predictions to smoldocling_doclaynet_results.jsonl')
print(f'\nExample prediction (first 500 chars):\n{doclaynet_results[0]["prediction_raw"][:500]}')

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

DocLayNet:   0%|          | 0/50 [00:00<?, ?it/s]

Saved 50 page predictions to smoldocling_doclaynet_results.jsonl

Example prediction (first 500 chars):
<doctag><page_footer><loc_6><loc_489><loc_10><loc_496>8</page_footer>
<text><loc_102><loc_15><loc_188><loc_35>Leigh Taliaferro, M.D. General Surgeon Abilene, Texas</text>
<text><loc_32><loc_54><loc_174><loc_62>Leigh Taliaferro, M.D., values consistency.</text>
<text><loc_32><loc_65><loc_193><loc_130>The Abilene native started his practice 17 years ago and has developed a flourishing business as a general surgeon. He estimates that 90 percent of his practice is for abdominal surgery. With such a 


In [49]:
# Look at 3 predictions end-to-end
with open('/content/smoldocling_doclaynet_results.jsonl') as f:
    lines = [json.loads(line) for line in f]

for i in [0, 5, 10]:
    print(f'--- Page {i} ---')
    print(lines[i]['prediction_raw'][:400])
    print()

--- Page 0 ---
<doctag><page_footer><loc_6><loc_489><loc_10><loc_496>8</page_footer>
<text><loc_102><loc_15><loc_188><loc_35>Leigh Taliaferro, M.D. General Surgeon Abilene, Texas</text>
<text><loc_32><loc_54><loc_174><loc_62>Leigh Taliaferro, M.D., values consistency.</text>
<text><loc_32><loc_65><loc_193><loc_130>The Abilene native started his practice 17 years ago and has developed a flourishing business as a 

--- Page 5 ---
<doctag><text><loc_42><loc_67><loc_457><loc_80>As of December 31, 2003, there remained 72,534 shares for which options may be granted in the future under the 1997 Stock Incentive Plan. The following table summarizes information about stock options outstanding at December 31, 2003:</text>
<otsl><loc_42><loc_83><loc_457><loc_155><ecel><ched>OPTIONS OUTSTANDING<lcel><lcel><ched>OPTIONS EXERCISABLE<lc

--- Page 10 ---
<doctag><section_header_level_1><loc_25><loc_25><loc_311><loc_38>NOTES TO THE FINANCIAL STATEMENTS</section_header_level_1>
<text><loc_25><loc_39><loc

## 1.5 - Dataset 3: PubTables-1M (table structure)

Task: cropped table image → OTSL structure.

Paper prompt: `Convert table to OTSL.`

Targets from the paper (Table 4): TEDS 0.65 with text, 0.88 structure-only.



In [23]:
#Input of the structure test set
PUBTABLES_IMG_URL = 'https://huggingface.co/datasets/bsmock/pubtables-1m/resolve/main/PubTables-1M-Structure_Images_Test.tar.gz'

os.makedirs('/content/pubtables_img', exist_ok=True)
print('Downloading PubTables-1M structure images (this may take a few minutes)...')
!wget -q -O /content/pubtables_img/images.tar.gz "{PUBTABLES_IMG_URL}"
!tar -xzf /content/pubtables_img/images.tar.gz -C /content/pubtables_img/
!rm /content/pubtables_img/images.tar.gz

# Find the images — directory structure varies, so search recursively
img_files = sorted(Path('/content/pubtables_img').rglob('*.jpg'))
if not img_files:
    img_files = sorted(Path('/content/pubtables_img').rglob('*.png'))
print(f'Found {len(img_files)} table images')

Found 93834 table images


In [25]:
N_PUBTABLES_SAMPLES = 100

pubtables_results = []
for img_path in tqdm(img_files[:N_PUBTABLES_SAMPLES], desc='PubTables'):
    image = Image.open(img_path).convert('RGB')

    # Run model with OTSL conversion prompt
    prediction = run_smoldocling(image, 'Convert table to OTSL.', max_tokens=1024)

    pubtables_results.append({
        'image_name': img_path.stem,   # Table ID, used to match to Phase 2 XML ground truth
        'image_path': str(img_path),
        'prediction_raw': prediction,
    })

df_pubtables = pd.DataFrame(pubtables_results)
df_pubtables.to_csv('/content/smoldocling_pubtables_results.csv', index=False)
print(f'Saved {len(df_pubtables)} table predictions to smoldocling_pubtables_results.csv')
print(f'\nSample OTSL output (first 500 chars):\n{pubtables_results[0]["prediction_raw"][:500]}')

PubTables:   0%|          | 0/100 [00:00<?, ?it/s]

Saved 100 table predictions to smoldocling_pubtables_results.csv

Sample OTSL output (first 500 chars):
<otsl><loc_0><loc_0><loc_500><loc_500><ched>Analytic dataset<ched>Main exposure<ched>African-American<lcel><ched>White<lcel><nl><ucel><ucel><ched>Cases (%)<ched>Controls (%)<ched>Cases (%)<ched>Controls (%)<nl><fcel>CBCS, entire<fcel>Birth order<fcel>335 (100.0)<fcel>332 (100.0)<fcel>526 (100.0)<fcel>458 (100.0)<nl><fcel>CBCS, born 1948 or later<fcel>131 (39.1)<fcel>135 (40.7)<fcel>235 (44.7)<fcel>181 (39.5)<ecel><nl><fcel>Maternal age dataset<fcel>Birth order, Maternal age<fcel>107 (31.9)<fcel>


In [27]:
# Quick quality check — how many got truncated?
df_pubtables['output_length'] = df_pubtables['prediction_raw'].str.len()
print(f'\nOutput length stats:')
print(df_pubtables['output_length'].describe())

# How many look complete (have closing </otsl> tag)?
complete = df_pubtables['prediction_raw'].str.contains('</otsl>').sum()
print(f'\nTables with complete OTSL output: {complete}/{len(df_pubtables)}')

# Look at a sample
print(f'\nSample prediction:\n{df_pubtables.iloc[0]["prediction_raw"][:400]}')


Output length stats:
count     100.000000
mean     1017.660000
std       662.462459
min       211.000000
25%       483.500000
50%       855.000000
75%      1407.250000
max      3191.000000
Name: output_length, dtype: float64

Tables with complete OTSL output: 98/100

Sample prediction:
<otsl><loc_0><loc_0><loc_500><loc_500><ched>Analytic dataset<ched>Main exposure<ched>African-American<lcel><ched>White<lcel><nl><ucel><ucel><ched>Cases (%)<ched>Controls (%)<ched>Cases (%)<ched>Controls (%)<nl><fcel>CBCS, entire<fcel>Birth order<fcel>335 (100.0)<fcel>332 (100.0)<fcel>526 (100.0)<fcel>458 (100.0)<nl><fcel>CBCS, born 1948 or later<fcel>131 (39.1)<fcel>135 (40.7)<fcel>235 (44.7)<fcel


## 1.6:  Saving to Google Drive


In [50]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
SAVE_DIR = '/content/drive/MyDrive/F26-04/phase3_results'
os.makedirs(SAVE_DIR, exist_ok=True)

for f in ['smoldocling_latex_results.csv',
          'smoldocling_doclaynet_results.jsonl',
          'smoldocling_pubtables_results.csv']:
    src = f'/content/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{SAVE_DIR}/{f}')
        print(f'Saved {f} to Drive')

print('\nAll Phase 3 predictions saved. Ready for Phase 4 metric computation.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved smoldocling_latex_results.csv to Drive
Saved smoldocling_doclaynet_results.jsonl to Drive
Saved smoldocling_pubtables_results.csv to Drive

All Phase 3 predictions saved. Ready for Phase 4 metric computation.


# Nougat as Baseline Comparison

Runs Nougat-base (350M) on the SAME samples SmolDocling was tested on, for fair side-by-side comparison in Phase 4.

**Two tasks (from paper Table 2):**
1. Im2LaTeX-230K — equation recognition (target F1: 0.60)
2. DocLayNet v1.2 — full-page OCR (target F1: 0.66)

**Not tested:** PubTables-1M. Nougat outputs Markdown tables which cannot be meaningfully compared against OTSL structure via TEDS.

## Known characteristics of Nougat
- Trained on academic papers (scholarly PDFs with LaTeX).
- Hallucinates on non-academic content or isolated image crops.
- Prone to repetition loops on complex layouts.
- These are EXPECTED behaviors that the paper's low F1 scores (0.60 / 0.66) already reflect.

# 2.1: Installing and setting nougat

In [28]:
!pip install -q -U transformers datasets Pillow tqdm pandas

import os, re, json, io, torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from pathlib import Path
from transformers import NougatProcessor, VisionEncoderDecoderModel
from datasets import load_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

MODEL_ID = 'facebook/nougat-base'
print(f'Loading {MODEL_ID}...')

processor = NougatProcessor.from_pretrained(MODEL_ID)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
model.eval()

processor.image_processor.do_crop_margin = False

print('Nougat loaded successfully')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 130.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 126.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you hav

preprocessor_config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/640 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

Nougat loaded successfully


## 2.2: Nougat inference function

Nougat has two well-known failure modes (documented in the paper's Table 2 scores):
1. **Repetition loops** — model gets stuck repeating the same tokens.
2. **Hallucination** — model invents content not in the image, especially on non-academic documents.

We mitigate repetition with `repetition_penalty=1.2` and `no_repeat_ngram_size=3`. Hallucination is a fundamental property of the model and cannot be fully suppressed and will be measured and qantified in phase 4.

In [34]:
def run_nougat(image, max_tokens=512):
    """Run Nougat on one image and return the generated Markdown/MMD output."""
    pixel_values = processor(
        image,
        return_tensors='pt',
        do_crop_margin=False,
        do_resize=True,
        size={"height": 896, "width": 672},
        resample=Image.BICUBIC,
        do_thumbnail=True,
        do_align_long_axis=False,
        do_rescale=True,
        rescale_factor=1/255,
        do_normalize=True,
        image_mean=[0.485, 0.456, 0.406],
        image_std=[0.229, 0.224, 0.225],
        do_pad=True,
    ).pixel_values.to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            min_length=1,
            max_new_tokens=max_tokens,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
        )

    sequence = processor.batch_decode(outputs, skip_special_tokens=True)[0]
    return sequence.strip()

def strip_mmd(text):
    """Remove Mathpix-Markdown formatting to get plain text for metrics."""
    text = re.sub(r'\\\(|\\\)|\\\[|\\\]', '', text)
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\*{1,2}([^*]+)\*{1,2}', r'\1', text)
    text = re.sub(r'\|', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# Sanity check
test_img = Image.new('RGB', (400, 200), 'white')
test_out = run_nougat(test_img, max_tokens=64)
print(f'Sanity check output: {test_out[:200]!r}')

Sanity check output: '* [15] M. Bershadsky, A. Khodjamirian and S. Yankielowicz, Phys. Rev. D **68**, 034902(R) (2003) [arXiv:hep-'


the output above shows nougat hallucainating on a simple white page


## 2.3: Dataset 1- Im2LaTeX: equation recognition


Same 50 images SmolDocling ran on. Nougat outputs LaTeX directly in its own format (often wrapped in `\(...\)`). We strip that for metric comparison.

In [35]:
# Same paths as SmolDocling notebook
LATEX_IMG_DIR = '/content/im2latex_data/PRINTED_TEX_230k/generated_png_images'
FORMULA_FILE = '/content/im2latex_data/PRINTED_TEX_230k/final_png_formulas.txt'
MAP_FILE = '/content/im2latex_data/PRINTED_TEX_230k/corresponding_png_images.txt'

# If the Kaggle data isn't already downloaded
if not os.path.exists(LATEX_IMG_DIR):
    from google.colab import userdata
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
    !kaggle datasets download -d gregoryeritsyan/im2latex-230k -p /content/im2latex_data/ --unzip -q

with open(FORMULA_FILE) as f:
    formulas = [line.strip() for line in f]
with open(MAP_FILE) as f:
    image_names = [line.strip() for line in f]

print(f'Loaded {len(formulas)} formula/image pairs')

Loaded 238329 formula/image pairs


In [37]:
# Matching the SmolDocling sample count (50) for fair comparison
N_SAMPLES = 50

nougat_latex_results = []
for img_name, formula in tqdm(list(zip(image_names, formulas))[:N_SAMPLES], desc='Nougat-Im2LaTeX'):
    img_path = os.path.join(LATEX_IMG_DIR, img_name)
    if not os.path.exists(img_path):
        continue

    image = Image.open(img_path).convert('RGB')

    try:
        prediction = run_nougat(image, max_tokens=256)   # Formulas are short
    except Exception as e:
        prediction = f'[ERROR: {str(e)[:80]}]'

    nougat_latex_results.append({
        'image': img_name,
        'ground_truth_latex': formula,
        'prediction_raw': prediction,
        'prediction_stripped': strip_mmd(prediction),
    })

df_nougat_latex = pd.DataFrame(nougat_latex_results)
df_nougat_latex.to_csv('/content/nougat_latex_results.csv', index=False)
print(f'\nSaved {len(df_nougat_latex)} predictions')
print(f'\nExamples (first 3):')
for i in range(min(10, len(df_nougat_latex))):
    print(f'\nGT:   {df_nougat_latex.iloc[i].ground_truth_latex[:100]}')
    print(f'Pred: {df_nougat_latex.iloc[i].prediction_stripped[:100]}')

Nougat-Im2LaTeX:   0%|          | 0/50 [00:00<?, ?it/s]


Saved 50 predictions

Examples (first 3):

GT:   R _ { 1 2 } K _ { 1 } R _ { 2 1 } d K _ { 2 } = d K _ { 2 } R _ { 1 2 } K _ { 1 } R _ { 1 2 } ^ { - 
Pred: R_{12}K_{4}R_

GT:   E _ { n } - E _ { m } = \frac { \lambda ^ { \prime } ( n ^ { 2 } y ^ { 2 } - m ^ { 2 } ) } { y ^ { 2
Pred: 

GT:   \sigma ^ { 1 } + i \sigma ^ { 2 } = f ( \sigma ^ { 1 } + i \sigma ^ { 2 } )
Pred: 

GT:   B | _ { \partial \Sigma _ { 3 } } \rightarrow B | _ { \partial \Sigma _ { 3 } } - \Lambda | _ { \par
Pred: 

GT:   \phi _ { i } ^ { \prime } ( x ) = \phi _ { i } ( x ) + \delta _ { \mathrm { B R S } } [ \phi _ { i }
Pred: d_{1}^{l}(x)=q_{i}(a)+q_

GT:   \partial _ { t } \bar { \sigma } _ { k } ^ { ( g f ) } = \frac { 1 } { \alpha } \partial _ { t } \ga
Pred: 

GT:   x _ { i } = y _ { i } \left( 1 - \eta _ { 1 } \eta _ { 2 } \right) , ~ ~ \sum _ { 0 } ^ { 3 } y _ { 
Pred: Figure Captions: * _Fig.1_ The energy dependence of the effective potential on the energy of a given

GT:   \delta e ^ { s A } = \int _ { 0 }

## 2.4: Dataset 2 DocLayNet: full-page OCR

Same 20 pages SmolDocling processed. We use `DocLayNet-v1.2` **test** split to match SmolDocling's evaluation exactly.

Expected as per research:
- Some pages will hallucinate content (Nougat invents text when unsure)
- Some pages will hit repetition loops despite repetition_penalty
- Academic-style pages will work reasonably well; business/patent/financial pages will suffer

This is consistent with the paper's reported F1 of 0.66 for Nougat on DocLayNet which will be quantified from our standing in phase 4

In [40]:
#matching SmolDocling's dataset exactly — v1.2, test split
N_SAMPLES = 50
ds = load_dataset('docling-project/DocLayNet-v1.2', split='test', streaming=True)

nougat_doclaynet_results = []
for i, example in enumerate(tqdm(ds, total=N_SAMPLES, desc='Nougat-DocLayNet')):
    if i >= N_SAMPLES:
        break

    image = example['image'].convert('RGB')

    try:
        prediction = run_nougat(image, max_tokens=1024)
    except Exception as e:
        prediction = f'[ERROR: {str(e)[:80]}]'

    nougat_doclaynet_results.append({
        'index': i,
        'prediction_raw': prediction,
        'prediction_stripped': strip_mmd(prediction),
    })

with open('/content/nougat_doclaynet_results.jsonl', 'w') as f:
    for res in nougat_doclaynet_results:
        f.write(json.dumps(res) + '\n')

print(f'\nSaved {len(nougat_doclaynet_results)} predictions')
print(f'\nSample prediction (first 400 chars):\n{nougat_doclaynet_results[0]["prediction_raw"][:400]}')

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Nougat-DocLayNet:   0%|          | 0/50 [00:00<?, ?it/s]


Saved 50 predictions

Sample prediction (first 400 chars):
Leigh Taliaferro, M.D., values consistency.

The Abilene native started his practice 17 years ago and has developed a flourishing business as a general surgeon. He estimates that 90 percent of his practice is for abdominal surgery. With such a busy practice, he finds comfort in having a reliable banking partner." I have almost every type of business, trust and personal account with First National 


## 2.5 Save all results to Drive

In [41]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
SAVE_DIR = '/content/drive/MyDrive/F26-04/phase3_results'
os.makedirs(SAVE_DIR, exist_ok=True)

for f in ['nougat_latex_results.csv', 'nougat_doclaynet_results.jsonl']:
    src = f'/content/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{SAVE_DIR}/{f}')
        print(f'Saved {f}')

print('\nNougat baseline predictions saved. Ready for Phase 4 comparison against SmolDocling.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved nougat_latex_results.csv
Saved nougat_doclaynet_results.jsonl

Nougat baseline predictions saved. Ready for Phase 4 comparison against SmolDocling.


##**SECTION 3**

# Table Transformer (TATR) Baseline — PubTables-1M

Runs Microsoft Table Transformer (DETR-based, 86M params) on the SAME 100 PubTables-1M test images SmolDocling ran on, and converts its bounding-box output to OTSL format so Phase 4 TEDS comparison is fair.

**Step:**
1. Run TATR to detect rows, columns, headers, and spans (as it outputs bounding boxes for 6 classes)
2. Build a grid by intersecting row bboxes with column bboxes
3. Mark header cells (from column header detections)
4. Mark spans (from spanning cell detections)
5. Serialize the grid as OTSL tokens: `<ched>`, `<fcel>`, `<lcel>`, `<ucel>`, `<nl>`'


# Note on paper comparison
The SmolDocling paper (Table 4) compares against **TableFormer** (IBM, 52M params), not TATR. TableFormer is harder to run (no HuggingFace checkpoint, requires the docling-ibm-models package with specific setup). TATR is the closest readily-available alternative and is itself a strong baseline being the state-of-the-art on PubTables-1M before TableFormer.

## 3.1:  Install and load TATR

In [51]:
!pip install -q -U transformers Pillow tqdm pandas

import os, re, json, torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from pathlib import Path
from transformers import TableTransformerForObjectDetection, DetrImageProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

# we use v1.1 structure recognition model trained on PubTables-1M
# this is the direct DETR-based baseline for table structure on PubTables-1M.
MODEL_ID = 'microsoft/table-transformer-structure-recognition'
print(f'Loading {MODEL_ID}...')

processor = DetrImageProcessor()
model = TableTransformerForObjectDetection.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

# The 6 classes TATR predicts — we map these to OTSL tokens later
TATR_CLASSES = {
    0: 'table',
    1: 'table column',
    2: 'table row',
    3: 'table column header',
    4: 'table projected row header',
    5: 'table spanning cell',
}
print('Table Transformer loaded successfully')


Using device: cuda
Loading microsoft/table-transformer-structure-recognition...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

TableTransformerForObjectDetection LOAD REPORT from: microsoft/table-transformer-structure-recognition
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Table Transformer loaded successfully


## 3.2 Inference + OTSL conversion

This is the bulk of the work. TATR gives us boxes (6 classes); we turn them into an OTSL string that mirrors SmolDocling's output format.

In [52]:
def run_tatr_detection(image, threshold=0.7):
    """
    Run TATR on an image, return dict of detections grouped by class.
    Each detection is {'bbox': [x1,y1,x2,y2], 'score': float}.
    """
    inputs = processor(image, return_tensors='pt').to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)

    # Post-process to get boxes in pixel coordinates
    target_sizes = torch.tensor([image.size[::-1]]).to(DEVICE)  # (H, W)
    results = processor.post_process_object_detection(
        outputs, threshold=threshold, target_sizes=target_sizes
    )[0]

    # Group detections by class label
    grouped = {c: [] for c in TATR_CLASSES.values()}
    for score, label, box in zip(results['scores'], results['labels'], results['boxes']):
        cls = TATR_CLASSES.get(label.item(), 'unknown')
        if cls in grouped:
            grouped[cls].append({
                'bbox': box.cpu().tolist(),
                'score': score.item(),
            })
    return grouped


def bbox_overlap_ratio(a, b):
    """Ratio of overlap area to min(area_a, area_b). Used to decide if a cell falls inside a row/column."""
    xa1, ya1, xa2, ya2 = a
    xb1, yb1, xb2, yb2 = b
    ix1, iy1 = max(xa1, xb1), max(ya1, yb1)
    ix2, iy2 = min(xa2, xb2), min(ya2, yb2)
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)
    return inter / max(1e-6, min(area_a, area_b))


def detections_to_otsl(detections):
    """
    Convert TATR detections to an OTSL string in the same format SmolDocling produces:
    <otsl><ched>...<nl><fcel>...<nl>...</otsl>

    Algorithm:
    1. Sort rows top-to-bottom, columns left-to-right.
    2. For each row×column intersection, create a cell.
    3. Mark cells that fall inside a 'column header' region as <ched>.
    4. Mark cells covered by 'spanning cell' boxes as spans (approximation: <lcel> for horizontal spans).
    5. Serialize grid as OTSL tokens.
    """
    rows = sorted(detections['table row'], key=lambda d: d['bbox'][1])       # top-to-bottom
    cols = sorted(detections['table column'], key=lambda d: d['bbox'][0])    # left-to-right
    headers = detections['table column header']                              # header strip(s)
    spans = detections['table spanning cell']                                # merged cells

    #TATR didn't find any rows or columns, return empty OTSL
    if not rows or not cols:
        return '<otsl></otsl>'

    # Build a grid: each cell is identified by (row_idx, col_idx)
    # Track which cells are headers and which are spans.
    tokens = ['<otsl>']

    for r_idx, row in enumerate(rows):
        row_bbox = row['bbox']
        # Is this entire row inside a header region
        row_cy = (row_bbox[1] + row_bbox[3]) / 2
        in_header = any(h['bbox'][1] <= row_cy <= h['bbox'][3] for h in headers)

        prev_col_spanned = False
        for c_idx, col in enumerate(cols):
            col_bbox = col['bbox']
            # The cell sits at the intersection of this row and column
            cell_bbox = [col_bbox[0], row_bbox[1], col_bbox[2], row_bbox[3]]

            #if any spanning-cell detection covers this grid cell strongly
            is_spanned = False
            for s in spans:
                if bbox_overlap_ratio(cell_bbox, s['bbox']) > 0.5:
                    is_spanned = True
                    break

            # Emit the OTSL token for this cell
            if is_spanned and prev_col_spanned:
                # Continuation of a horizontal span — left-looking cell
                tokens.append('<lcel>')
            elif in_header:
                tokens.append('<ched>')
                prev_col_spanned = is_spanned
            else:
                tokens.append('<fcel>')
                prev_col_spanned = is_spanned

        tokens.append('<nl>')

    tokens.append('</otsl>')
    return ''.join(tokens)


# Sanity check
test_img = Image.new('RGB', (400, 200), 'white')
test_detections = run_tatr_detection(test_img)
print('Detection counts on blank image:')
for cls, dets in test_detections.items():
    print(f'  {cls}: {len(dets)}')
print(f'\nOTSL output: {detections_to_otsl(test_detections)}')

Detection counts on blank image:
  table: 1
  table column: 2
  table row: 13
  table column header: 0
  table projected row header: 0
  table spanning cell: 0

OTSL output: <otsl><fcel><fcel><nl><fcel><fcel><nl><fcel><fcel><nl><fcel><fcel><nl><fcel><fcel><nl><fcel><fcel><nl><fcel><fcel><nl><fcel><fcel><nl><fcel><fcel><nl><fcel><fcel><nl><fcel><fcel><nl><fcel><fcel><nl><fcel><fcel><nl></otsl>


## 3.3 — Run on PubTables-1M test images

Uses the same 100 images we ran SmolDocling on. If the image directory doesn't exist yet (fresh runtime), re-download the archive.

In [ ]:
PUBTABLES_IMG_DIR = '/content/pubtables_img'
import os
from pathlib import Path
# Re-download if not present (fresh runtime)
if not os.path.exists(PUBTABLES_IMG_DIR) or not any(Path(PUBTABLES_IMG_DIR).rglob('*.jpg')):
    PUBTABLES_IMG_URL = 'https://huggingface.co/datasets/bsmock/pubtables-1m/resolve/main/PubTables-1M-Structure_Images_Test.tar.gz'
    os.makedirs(PUBTABLES_IMG_DIR, exist_ok=True)
    print('Downloading PubTables-1M images...')
    !wget -q -O /content/pubtables_img/images.tar.gz "{PUBTABLES_IMG_URL}"
    !tar -xzf /content/pubtables_img/images.tar.gz -C /content/pubtables_img/
    !rm /content/pubtables_img/images.tar.gz

img_files = sorted(Path(PUBTABLES_IMG_DIR).rglob('*.jpg'))
if not img_files:
    img_files = sorted(Path(PUBTABLES_IMG_DIR).rglob('*.png'))
print(f'Found {len(img_files)} table images')

In [ ]:
# Match SmolDocling's 100 samples exactly — same first 100 images, same order
N_SAMPLES = 100

tatr_results = []
for img_path in tqdm(img_files[:N_SAMPLES], desc='TATR-PubTables'):
    try:
        image = Image.open(img_path).convert('RGB')
        detections = run_tatr_detection(image, threshold=0.7)
        otsl = detections_to_otsl(detections)

        # Count what TATR detected — useful for debugging and the report
        n_rows = len(detections['table row'])
        n_cols = len(detections['table column'])
        n_headers = len(detections['table column header'])
        n_spans = len(detections['table spanning cell'])
    except Exception as e:
        otsl = f'<otsl>[ERROR: {str(e)[:60]}]</otsl>'
        n_rows = n_cols = n_headers = n_spans = 0

    tatr_results.append({
        'image_name': img_path.stem,
        'image_path': str(img_path),
        'prediction_raw': otsl,
        'n_rows_detected': n_rows,
        'n_cols_detected': n_cols,
        'n_headers_detected': n_headers,
        'n_spans_detected': n_spans,
    })

df_tatr = pd.DataFrame(tatr_results)
df_tatr.to_csv('/content/tatr_pubtables_results.csv', index=False)
print(f'\nSaved {len(df_tatr)} predictions to tatr_pubtables_results.csv')

## 3.4 Inspect and save

Compare TATR's grid reconstruction against SmolDocling's direct OTSL generation.

In [ ]:
# Detection statistics
print(f'Average rows detected:    {df_tatr["n_rows_detected"].mean():.1f}')
print(f'Average columns detected: {df_tatr["n_cols_detected"].mean():.1f}')
print(f'Tables with headers:      {(df_tatr["n_headers_detected"] > 0).sum()}/{len(df_tatr)}')
print(f'Tables with spans:        {(df_tatr["n_spans_detected"] > 0).sum()}/{len(df_tatr)}')
print(f'Tables with 0 rows:       {(df_tatr["n_rows_detected"] == 0).sum()}/{len(df_tatr)}  (TATR failure cases)')

# Output length stats
df_tatr['otsl_length'] = df_tatr['prediction_raw'].str.len()
print(f'\nOTSL length — min: {df_tatr["otsl_length"].min()}, median: {df_tatr["otsl_length"].median():.0f}, max: {df_tatr["otsl_length"].max()}')

# Show a sample
print(f'\nSample TATR OTSL output:')
print(df_tatr.iloc[0]['prediction_raw'][:400])

##**3.5 Save to Drive**

In [ ]:
# Save to Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
SAVE_DIR = '/content/drive/MyDrive/F26-04/phase3_results'
os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copy('/content/tatr_pubtables_results.csv', f'{SAVE_DIR}/tatr_pubtables_results.csv')
print(f'Saved TATR results to Drive ({SAVE_DIR})')